# Module 3: Production Deployment with Amazon Bedrock AgentCore

## Overview

In this module, you'll deploy the **Product Catalog Agent** from Module 01 to production using **Amazon Bedrock AgentCore**. The local prototype's mock identity (`UserSession`) maps to real JWT-based authentication, and the MCP server becomes a Lambda function behind AgentCore Gateway. **Observability is enabled from day one** using OpenTelemetry instrumentation with CloudWatch GenAI Observability.

### What You'll Build

```
                     ┌──────────────────────────────────────────────┐
                     │           Cognito User Pool                  │
                     │  ┌────────────┐    ┌────────────┐           │
                     │  │  customer   │    │   admin    │           │
                     │  │   group     │    │   group    │           │
                     │  └────────────┘    └────────────┘           │
                     └───────────────┬──────────────────────────────┘
                                     │ JWT (cognito:groups)
                                     ▼
┌──────────┐  invoke  ┌──────────────────────────────────┐
│ Streamlit │ ──────► │     AgentCore Runtime             │
│   App     │         │  ┌────────────────────────────┐   │
└──────────┘          │  │  Product Catalog Agent      │   │  ──► OTEL Traces
                      │  │  • JWT role extraction       │   │      (X-Ray + CloudWatch)
                      │  │  • Tool filtering by role    │   │
                      │  │  • Role-aware system prompt  │   │
                      │  └──────────┬─────────────────┘   │
                      └─────────────┼─────────────────────┘
                                    │ SigV4 + Bearer token
                                    ▼
                      ┌─────────────────────────────────────────┐
                      │        AgentCore Gateway                 │
                      │   ┌─────────────────────────────────┐   │
                      │   │  RBAC Interceptor Lambda         │   │
                      │   │  • REQUEST: blocks admin tools   │   │
                      │   │    for customer role              │   │
                      │   │  • RESPONSE: filters tools/list  │   │
                      │   │    to hide admin tools            │   │
                      │   └─────────────────────────────────┘   │
                      │                                         │
                      │   ┌─────────────────────────────────┐   │
                      │   │  ProductTools Target (Lambda)    │   │
                      │   │  11 MCP tools via Lambda         │   │
                      │   └───────────────┬─────────────────┘   │
                      └───────────────────┼─────────────────────┘
                                          │
                                          ▼
                      ┌───────────────────────────────────┐
                      │      DynamoDB Products Table      │
                      └───────────────────────────────────┘
```

### Prototype to Production Mapping

| Module 01 (Prototype) | Module 03 (Production) |
|----------------------|------------------------|
| `UserSession` class | Cognito JWT token |
| `role` field | `cognito:groups` claim |
| Local MCP server (stdio) | Lambda behind AgentCore Gateway |
| Local tool filtering | **Defense-in-depth**: Agent filtering + Gateway interceptor |
| `python agent.py` | AgentCore Runtime (container) |
| No observability | **Full OTEL tracing** + GenAI events in CloudWatch |

### Defense-in-Depth RBAC

Two independent layers enforce access control:
1. **Agent-side** (same as Module 01): Tool filtering removes admin tools from customer agents
2. **Gateway interceptor** (new): Lambda intercepts requests/responses at infrastructure level

### Learning Objectives
1. Deploy Lambda functions as MCP tool backends via AgentCore Gateway
2. Configure Cognito User Pool with groups for JWT-based RBAC
3. Implement a Gateway interceptor for infrastructure-level access control
4. Build and deploy a containerized agent to AgentCore Runtime
5. **Configure OpenTelemetry observability** with CloudWatch GenAI Observability
6. Test role-based access control end-to-end in production
7. **Validate traces and GenAI events** in CloudWatch

### Prerequisites
- Module 01 completed (DynamoDB products table populated)
- AWS credentials with AgentCore, Lambda, Cognito, ECR, CloudWatch, and IAM permissions
- Docker installed (for building container images)

### Time: ~45 minutes

## Step 1: Environment Setup

In [1]:
import boto3
import json
import os
import time
import uuid

# Load region from previous module or use default
try:
    %store -r REGION
    print(f"Loaded REGION from previous module: {REGION}")
except:
    session = boto3.Session()
    REGION = session.region_name or 'us-west-2'
    print(f"Using default region: {REGION}")

os.environ['AWS_REGION'] = REGION
os.environ['AWS_DEFAULT_REGION'] = REGION

# Workshop configuration
WORKSHOP_PREFIX = 'ecommerce-workshop'
PRODUCTS_TABLE = f'{WORKSHOP_PREFIX}-products'

# Naming conventions
LAMBDA_ROLE_NAME = f'{WORKSHOP_PREFIX}-lambda-role'
GATEWAY_ROLE_NAME = f'{WORKSHOP_PREFIX}-gateway-role'
GATEWAY_NAME = f'{WORKSHOP_PREFIX}-product-gateway'
RUNTIME_ROLE_NAME = f'{WORKSHOP_PREFIX}-runtime-role'
RUNTIME_NAME = 'ecommerce_workshop_product_catalog_agent'  # underscores only - AgentCore Runtime name constraint
COGNITO_POOL_NAME = f'{WORKSHOP_PREFIX}-user-pool'
ECR_REPO_NAME = f'{WORKSHOP_PREFIX}-product-catalog-agent'
TEST_PASSWORD = 'Workshop1234!'

print(f"\nConfiguration:")
print(f"  Region: {REGION}")
print(f"  Products Table: {PRODUCTS_TABLE}")
print(f"  Gateway: {GATEWAY_NAME}")

Loaded REGION from previous module: us-west-2

Configuration:
  Region: us-west-2
  Products Table: ecommerce-workshop-products
  Gateway: ecommerce-workshop-product-gateway


In [2]:
# Verify AWS credentials
sts = boto3.client('sts', region_name=REGION)
identity = sts.get_caller_identity()
ACCOUNT_ID = identity['Account']

print(f"AWS Account: {ACCOUNT_ID}")
print(f"AWS Identity: {identity['Arn']}")

AWS Account: 534409838809
AWS Identity: arn:aws:iam::534409838809:user/Admin


## Step 2: Create IAM Roles

We need three IAM roles:
1. **Lambda execution role** - Allows Lambda functions to access DynamoDB
2. **Gateway role** - Allows AgentCore Gateway to invoke Lambda functions
3. **Runtime role** - Allows AgentCore Runtime to invoke Bedrock models and connect to Gateway

In [3]:
from utils import create_lambda_execution_role

iam_client = boto3.client('iam', region_name=REGION)

# DynamoDB table ARN
products_table_arn = f'arn:aws:dynamodb:{REGION}:{ACCOUNT_ID}:table/{PRODUCTS_TABLE}'

# Create Lambda execution role
print("Creating Lambda execution role...")
lambda_role_resp = create_lambda_execution_role(
    iam_client,
    LAMBDA_ROLE_NAME,
    [products_table_arn]
)
LAMBDA_ROLE_ARN = lambda_role_resp['Role']['Arn']
print(f"Lambda Role ARN: {LAMBDA_ROLE_ARN}")

Creating Lambda execution role...
Role ecommerce-workshop-lambda-role already exists, retrieving ARN...
Lambda Role ARN: arn:aws:iam::534409838809:role/ecommerce-workshop-lambda-role


## Step 3: Deploy Lambda Functions

We deploy two Lambda functions:
1. **Product Tools Lambda** - Contains all 11 MCP tools (same tools as Module 01's MCP server)
2. **RBAC Interceptor Lambda** - Gateway interceptor that enforces role-based access control

### Product Tools Lambda
This Lambda receives tool calls from AgentCore Gateway and routes them to the appropriate handler. The tool name comes from `context.client_context.custom['bedrockAgentCoreToolName']`.

### RBAC Interceptor Lambda
This Lambda runs at two interception points on the Gateway:
- **REQUEST**: Blocks `tools/call` for admin tools when the caller is a customer
- **RESPONSE**: Filters `tools/list` to hide admin tools from customer sessions

In [4]:
from utils import create_lambda_function

lambda_client = boto3.client('lambda', region_name=REGION)

# Environment variables for product tools Lambda
lambda_env_vars = {
    'PRODUCTS_TABLE_NAME': PRODUCTS_TABLE
}

# Deploy Product Tools Lambda (11 tools)
print("Deploying Product Tools Lambda...")
product_tools_result = create_lambda_function(
    lambda_client,
    f'{WORKSHOP_PREFIX}-product-tools',
    LAMBDA_ROLE_ARN,
    'lambda_tools/product_tools_lambda.py',
    'product_tools_lambda.lambda_handler',
    lambda_env_vars,
    REGION
)
PRODUCT_TOOLS_ARN = product_tools_result['function_arn']
print(f"Product Tools ARN: {PRODUCT_TOOLS_ARN}\n")

# Deploy RBAC Interceptor Lambda
print("Deploying RBAC Interceptor Lambda...")
interceptor_result = create_lambda_function(
    lambda_client,
    f'{WORKSHOP_PREFIX}-rbac-interceptor',
    LAMBDA_ROLE_ARN,
    'lambda_tools/rbac_interceptor_lambda.py',
    'rbac_interceptor_lambda.lambda_handler',
    {},  # No env vars needed - interceptor is stateless
    REGION
)
INTERCEPTOR_ARN = interceptor_result['function_arn']
print(f"RBAC Interceptor ARN: {INTERCEPTOR_ARN}")

Deploying Product Tools Lambda...
Function ecommerce-workshop-product-tools exists, updating...
Product Tools ARN: arn:aws:lambda:us-west-2:534409838809:function:ecommerce-workshop-product-tools

Deploying RBAC Interceptor Lambda...
Function ecommerce-workshop-rbac-interceptor exists, updating...
RBAC Interceptor ARN: arn:aws:lambda:us-west-2:534409838809:function:ecommerce-workshop-rbac-interceptor


## Step 4: Set Up Cognito User Pool with Groups

In Module 01, we used a `UserSession` class with a `role` field. In production, user roles come from **Cognito groups** embedded in the JWT token's `cognito:groups` claim.

We create:
- A Cognito User Pool with `customer` and `admin` groups
- An app client for user login (USER_PASSWORD_AUTH for the Streamlit app)
- An app client for M2M authentication (client_credentials for Gateway access)
- Test users: one customer, one admin

In [5]:
from utils import (
    get_or_create_cognito_user_pool,
    create_cognito_group,
    get_or_create_cognito_resource_server,
    get_or_create_cognito_app_client,
    get_or_create_user_app_client,
    get_or_create_cognito_domain,
    create_test_user
)

cognito_client = boto3.client('cognito-idp', region_name=REGION)

# 1. Create User Pool
print("Setting up Cognito User Pool...")
USER_POOL_ID = get_or_create_cognito_user_pool(cognito_client, COGNITO_POOL_NAME)

# 2. Create groups for RBAC
print("\nCreating RBAC groups...")
create_cognito_group(cognito_client, USER_POOL_ID, 'customer', 'Read-only product catalog access')
create_cognito_group(cognito_client, USER_POOL_ID, 'admin', 'Full product catalog access (read + write)')

# 3. Create resource server and M2M app client (for Gateway JWT validation)
COGNITO_RESOURCE_SERVER_ID = f'{WORKSHOP_PREFIX}-gateway-api'
SCOPES = [
    {"ScopeName": "gateway:read", "ScopeDescription": "Read access to Gateway tools"},
    {"ScopeName": "gateway:write", "ScopeDescription": "Write access to Gateway tools"}
]
get_or_create_cognito_resource_server(
    cognito_client, USER_POOL_ID, COGNITO_RESOURCE_SERVER_ID,
    f"{WORKSHOP_PREFIX} Gateway API", SCOPES
)

M2M_CLIENT_NAME = f'{WORKSHOP_PREFIX}-gateway-client'
M2M_CLIENT_ID, M2M_CLIENT_SECRET = get_or_create_cognito_app_client(
    cognito_client, USER_POOL_ID, M2M_CLIENT_NAME, COGNITO_RESOURCE_SERVER_ID, SCOPES
)

# 4. Create user-facing app client (for Streamlit login)
USER_CLIENT_NAME = f'{WORKSHOP_PREFIX}-user-client'
USER_CLIENT_ID = get_or_create_user_app_client(
    cognito_client, USER_POOL_ID, USER_CLIENT_NAME
)

# 5. Create Cognito domain for OAuth
DOMAIN_PREFIX = f"{WORKSHOP_PREFIX}-{str(uuid.uuid4())[:8]}"
get_or_create_cognito_domain(cognito_client, USER_POOL_ID, DOMAIN_PREFIX)

# 6. Create test users with group membership
print("\nCreating test users...")
CUSTOMER_EMAIL = 'john.customer@example.com'
ADMIN_EMAIL = 'alice.admin@example.com'

create_test_user(
    cognito_client, USER_POOL_ID, CUSTOMER_EMAIL, TEST_PASSWORD,
    group_name='customer', name='John Smith'
)
create_test_user(
    cognito_client, USER_POOL_ID, ADMIN_EMAIL, TEST_PASSWORD,
    group_name='admin', name='Alice Admin'
)

# Discovery URL for JWT validation
COGNITO_DISCOVERY_URL = f'https://cognito-idp.{REGION}.amazonaws.com/{USER_POOL_ID}/.well-known/openid-configuration'

print(f"\nCognito Configuration:")
print(f"  User Pool ID: {USER_POOL_ID}")
print(f"  Groups: customer, admin")
print(f"  M2M Client ID: {M2M_CLIENT_ID}")
print(f"  User Client ID: {USER_CLIENT_ID}")
print(f"  Test Customer: {CUSTOMER_EMAIL}")
print(f"  Test Admin: {ADMIN_EMAIL}")

Setting up Cognito User Pool...
Found existing user pool: us-west-2_zZFgcKaqx

Creating RBAC groups...
Group 'customer' already exists
Group 'admin' already exists
Resource server ecommerce-workshop-gateway-api already exists
Found existing app client: 3ghf0qftc5fqvttsaussfhir83
Found existing user app client: 3nvfo8a3bfvipd9k07pfi37mnf
Updated auth flows on user app client
Domain ecommerce-workshop-8fd5a5cf already exists or invalid

Creating test users...
User john.customer@example.com already exists
Added john.customer@example.com to group: customer
User alice.admin@example.com already exists
Added alice.admin@example.com to group: admin

Cognito Configuration:
  User Pool ID: us-west-2_zZFgcKaqx
  Groups: customer, admin
  M2M Client ID: 3ghf0qftc5fqvttsaussfhir83
  User Client ID: 3nvfo8a3bfvipd9k07pfi37mnf
  Test Customer: john.customer@example.com
  Test Admin: alice.admin@example.com


## Step 5: Create AgentCore Gateway with RBAC Interceptor

The AgentCore Gateway exposes our Lambda tools as MCP endpoints. We configure it with:
- **JWT authorizer** using Cognito for authentication
- **RBAC interceptor** Lambda for access control enforcement

The interceptor runs at two points:
1. **REQUEST**: Before a tool is called - blocks admin tools for customer role
2. **RESPONSE**: After tools/list - filters out admin tools for customer role

In [6]:
from utils import create_agentcore_gateway_role, create_gateway

gateway_client = boto3.client('bedrock-agentcore-control', region_name=REGION)

# Create Gateway IAM role (needs permission to invoke both Lambda functions)
print("Creating Gateway IAM role...")
gateway_role_resp = create_agentcore_gateway_role(
    iam_client,
    GATEWAY_ROLE_NAME,
    [PRODUCT_TOOLS_ARN, INTERCEPTOR_ARN]
)
GATEWAY_ROLE_ARN = gateway_role_resp['Role']['Arn']
print(f"Gateway Role ARN: {GATEWAY_ROLE_ARN}")

Creating Gateway IAM role...
Role ecommerce-workshop-gateway-role already exists, updating policy...
Updated Lambda invoke policy on Gateway role
Gateway Role ARN: arn:aws:iam::534409838809:role/ecommerce-workshop-gateway-role


In [7]:
# Create Gateway with JWT auth and RBAC interceptor
# allowedClients validates client_id claim in JWT access tokens
# Both M2M and user app clients are allowed (user access tokens have client_id = USER_CLIENT_ID)
auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [M2M_CLIENT_ID, USER_CLIENT_ID],
        "discoveryUrl": COGNITO_DISCOVERY_URL
    }
}

print("Creating AgentCore Gateway with RBAC interceptor...")
gateway_response = create_gateway(
    gateway_client,
    GATEWAY_NAME,
    GATEWAY_ROLE_ARN,
    auth_config,
    'Product Catalog Gateway with RBAC interceptor',
    interceptor_lambda_arn=INTERCEPTOR_ARN  # Attach RBAC interceptor
)

if gateway_response:
    GATEWAY_ID = gateway_response['gatewayId']
    GATEWAY_URL = gateway_response['gatewayUrl']
    GATEWAY_ARN = f'arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:gateway/{GATEWAY_ID}'
    print(f"\nGateway Created:")
    print(f"  Gateway ID: {GATEWAY_ID}")
    print(f"  Gateway URL: {GATEWAY_URL}")
    print(f"  RBAC Interceptor: Attached (REQUEST + RESPONSE)")
else:
    print("ERROR: Gateway creation failed")

Creating AgentCore Gateway with RBAC interceptor...
Gateway 'ecommerce-workshop-product-gateway' already exists, retrieving...
Found existing Gateway: ecommerce-workshop-product-gateway-2vhjfcwgfy

Gateway Created:
  Gateway ID: ecommerce-workshop-product-gateway-2vhjfcwgfy
  Gateway URL: https://ecommerce-workshop-product-gateway-2vhjfcwgfy.gateway.bedrock-agentcore.us-west-2.amazonaws.com/mcp
  RBAC Interceptor: Attached (REQUEST + RESPONSE)


## Step 6: Register Product Tools as Gateway Target

We register all 11 product tools (6 read + 5 admin) as a single Lambda target on the Gateway. The tool schemas tell the Gateway how to invoke each tool.

In [8]:
from utils import create_lambda_gateway_target, get_product_tool_schemas

# Get all 11 tool schemas
TOOL_SCHEMAS = get_product_tool_schemas()
print(f"Registering {len(TOOL_SCHEMAS)} product tools as Gateway target:\n")
for schema in TOOL_SCHEMAS:
    print(f"  - {schema['name']}: {schema['description'][:60]}...")

# Create the Lambda target
print(f"\nCreating Gateway target...")
product_target = create_lambda_gateway_target(
    gateway_client,
    GATEWAY_ID,
    'ProductTools',
    PRODUCT_TOOLS_ARN,
    TOOL_SCHEMAS,
    'Product catalog tools - 6 read tools + 5 admin tools'
)

if product_target:
    print(f"Target created: ProductTools")
    print(f"Target ID: {product_target.get('targetId')}")
else:
    print("ERROR: Failed to create Gateway target")

Registering 11 product tools as Gateway target:

  - search_products: Search for products in the catalog using keywords and option...
  - get_product_details: Get detailed information about a specific product by its ID...
  - check_inventory: Check inventory availability and stock quantity for a produc...
  - get_product_recommendations: Get product recommendations based on category and price crit...
  - compare_products: Compare multiple products side by side (2-5 products)...
  - get_return_policy: Get return policy information, optionally for a specific pro...
  - create_product: Create a new product in the catalog (admin only)...
  - update_product: Update an existing product's information (admin only)...
  - delete_product: Soft-delete a product by marking it as discontinued (admin o...
  - update_inventory: Update inventory levels for a product (admin only)...
  - update_pricing: Update pricing for a product, optionally set a sale price (a...

Creating Gateway target...
Target 'P

## Step 7: Test Gateway Connection

Before deploying the agent container, let's verify the Gateway works by connecting directly with MCP and testing tool discovery for different roles.

In [9]:
# Wait for Gateway to be ready
print("Waiting for Gateway resources to propagate...")
time.sleep(15)

# Get an M2M OAuth token for Gateway access
from utils import get_oauth_token

SCOPE_STRING = f"{COGNITO_RESOURCE_SERVER_ID}/gateway:read {COGNITO_RESOURCE_SERVER_ID}/gateway:write"

print("Getting OAuth token from Cognito...")
token_response = get_oauth_token(
    USER_POOL_ID, M2M_CLIENT_ID, M2M_CLIENT_SECRET, SCOPE_STRING, REGION
)

if 'access_token' in token_response:
    ACCESS_TOKEN = token_response['access_token']
    print(f"Access token obtained (expires in {token_response.get('expires_in', 'unknown')}s)")
else:
    print(f"Failed to get token: {token_response}")

Waiting for Gateway resources to propagate...
Getting OAuth token from Cognito...
Access token obtained (expires in 3600s)


In [10]:
# Test MCP connection to Gateway - list tools
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamablehttp_client

print("Testing MCP connection to Gateway...\n")

def create_mcp_transport():
    return streamablehttp_client(
        GATEWAY_URL,
        headers={"Authorization": f"Bearer {ACCESS_TOKEN}"}
    )

mcp_client = MCPClient(create_mcp_transport)

with mcp_client:
    tools = mcp_client.list_tools_sync()
    print(f"Found {len(tools)} MCP tools via Gateway:\n")
    for tool in tools:
        print(f"  - {tool.tool_name}")

print("\nGateway connection successful!")

Testing MCP connection to Gateway...

Found 6 MCP tools via Gateway:

  - ProductTools___check_inventory
  - ProductTools___compare_products
  - ProductTools___get_product_details
  - ProductTools___get_product_recommendations
  - ProductTools___get_return_policy
  - ProductTools___search_products

Gateway connection successful!


## Step 8: Test RBAC with Different User Tokens

Let's verify the RBAC interceptor works by getting tokens for both user roles and checking which tools each sees.

In [11]:
from utils import get_user_token

# Get tokens for both test users
print("Getting JWT tokens for test users...\n")

customer_tokens = get_user_token(
    cognito_client, USER_POOL_ID, USER_CLIENT_ID,
    CUSTOMER_EMAIL, TEST_PASSWORD
)

admin_tokens = get_user_token(
    cognito_client, USER_POOL_ID, USER_CLIENT_ID,
    ADMIN_EMAIL, TEST_PASSWORD
)

# Decode and show claims
import base64

def decode_jwt_claims(token):
    parts = token.split('.')
    payload_b64 = parts[1]
    padding = 4 - len(payload_b64) % 4
    if padding != 4:
        payload_b64 += '=' * padding
    return json.loads(base64.urlsafe_b64decode(payload_b64))

if customer_tokens.get('id_token'):
    claims = decode_jwt_claims(customer_tokens['id_token'])
    print(f"Customer JWT claims:")
    print(f"  email: {claims.get('email')}")
    print(f"  cognito:groups: {claims.get('cognito:groups', [])}")

if admin_tokens.get('id_token'):
    claims = decode_jwt_claims(admin_tokens['id_token'])
    print(f"\nAdmin JWT claims:")
    print(f"  email: {claims.get('email')}")
    print(f"  cognito:groups: {claims.get('cognito:groups', [])}")

Getting JWT tokens for test users...

Got tokens for user: john.customer@example.com
Got tokens for user: alice.admin@example.com
Customer JWT claims:
  email: john.customer@example.com
  cognito:groups: ['customer']

Admin JWT claims:
  email: alice.admin@example.com
  cognito:groups: ['admin']


> **Note**: The RBAC interceptor reads the `Authorization` header's JWT token to determine the caller's role. The Gateway itself validates JWT signatures via the Cognito discovery URL. The interceptor only needs to decode the payload (no verification needed since the Gateway already validated it).

## Step 9: Build and Push Agent Container

AgentCore Runtime runs agents as containers. We'll build a Docker image for the Product Catalog Agent and push it to Amazon ECR.

The agent container:
- Uses `BedrockAgentCoreApp` as the entrypoint
- Connects to Gateway via SigV4-signed MCP client
- Extracts user role from JWT `cognito:groups` claim
- Filters tools by role (same pattern as Module 01)
- Runs on port 8080 with OpenTelemetry instrumentation

In [12]:
import subprocess

# Create ECR repository
ecr_client = boto3.client('ecr', region_name=REGION)

print("Creating ECR repository...")
try:
    ecr_client.create_repository(
        repositoryName=ECR_REPO_NAME,
        imageScanningConfiguration={'scanOnPush': True},
        imageTagMutability='MUTABLE'
    )
    print(f"Created: {ECR_REPO_NAME}")
except ecr_client.exceptions.RepositoryAlreadyExistsException:
    print(f"Exists: {ECR_REPO_NAME}")

ECR_REGISTRY = f"{ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com"
CONTAINER_URI = f"{ECR_REGISTRY}/{ECR_REPO_NAME}:latest"
print(f"Container URI: {CONTAINER_URI}")

Creating ECR repository...
Exists: ecommerce-workshop-product-catalog-agent
Container URI: 534409838809.dkr.ecr.us-west-2.amazonaws.com/ecommerce-workshop-product-catalog-agent:latest


In [13]:
# Login to ECR
print("Logging into ECR...")
login_cmd = f"aws ecr get-login-password --region {REGION} | docker login --username AWS --password-stdin {ECR_REGISTRY}"
result = subprocess.run(login_cmd, shell=True, capture_output=True, text=True)
if result.returncode == 0:
    print("ECR login successful")
else:
    print(f"ECR login failed: {result.stderr}")

Logging into ECR...
ECR login successful


In [14]:
# Build Docker image (ARM64 required by AgentCore Runtime)
print("Building Docker image (ARM64)...")
print("This may take a few minutes on first build.\n")

build_cmd = f"docker build --platform linux/arm64 -t {ECR_REPO_NAME}:latest agents/"
result = subprocess.run(build_cmd, shell=True, capture_output=True, text=True)

if result.returncode == 0:
    print("Build successful!")
else:
    print(f"Build failed:\n{result.stderr}")

Building Docker image (ARM64)...
This may take a few minutes on first build.

Build successful!


In [15]:
# Tag and push to ECR
print("Pushing to ECR...")

tag_cmd = f"docker tag {ECR_REPO_NAME}:latest {CONTAINER_URI}"
subprocess.run(tag_cmd, shell=True, check=True)

push_cmd = f"docker push {CONTAINER_URI}"
result = subprocess.run(push_cmd, shell=True, capture_output=True, text=True)

if result.returncode == 0:
    print(f"Pushed successfully: {CONTAINER_URI}")
else:
    print(f"Push failed:\n{result.stderr}")

Pushing to ECR...
Pushed successfully: 534409838809.dkr.ecr.us-west-2.amazonaws.com/ecommerce-workshop-product-catalog-agent:latest


## Step 10: Create AgentCore Runtime with Observability

Deploy the containerized agent to AgentCore Runtime. The runtime provides:
- Managed container hosting with auto-scaling
- **OpenTelemetry-based observability** with CloudWatch GenAI Observability
- Secure endpoint for agent invocation

### Observability Configuration

The agent container uses `BedrockAgentCoreApp` with `opentelemetry-instrument` as the entrypoint. We configure OTEL environment variables so that:
- **Traces** are exported to AWS X-Ray via OTLP
- **Logs** (GenAI events) are exported to CloudWatch Logs via OTLP
- **GenAI semantic conventions** (`gen_ai_tool_definitions`) capture detailed tool call and LLM invocation events

After deploying the runtime, we'll configure CloudWatch Log Delivery to route telemetry data to the appropriate destinations.

In [16]:
from utils import create_agent_runtime_role

# Create Runtime IAM role with Gateway invocation permission
print("Creating AgentCore Runtime role...")
runtime_role_resp = create_agent_runtime_role(
    iam_client,
    RUNTIME_ROLE_NAME,
    gateway_arn=GATEWAY_ARN  # Permission to invoke Gateway tools
)
RUNTIME_ROLE_ARN = runtime_role_resp['Role']['Arn']
print(f"Runtime Role ARN: {RUNTIME_ROLE_ARN}")

Creating AgentCore Runtime role...
Role ecommerce-workshop-runtime-role already exists, updating policies...
Updated gateway policy on runtime role
Ensured X-Ray tracing policy on runtime role
Runtime Role ARN: arn:aws:iam::534409838809:role/ecommerce-workshop-runtime-role


In [17]:
from utils import create_agent_runtime

agentcore_client = boto3.client('bedrock-agentcore-control', region_name=REGION)

# Derive OTEL service name from runtime name (underscores -> dashes)
OTEL_SERVICE_NAME = RUNTIME_NAME.replace('_', '-')

# Agent environment variables — includes OTEL configuration for observability
agent_env_vars = {
    # Application configuration
    'AGENT_REGION': REGION,
    'GATEWAY_URL': GATEWAY_URL,
    'MODEL_ID': 'global.anthropic.claude-haiku-4-5-20251001-v1:0',

    # OpenTelemetry configuration for CloudWatch GenAI Observability
    'AGENT_OBSERVABILITY_ENABLED': 'true',
    'OTEL_PYTHON_DISTRO': 'aws_distro',
    'OTEL_PYTHON_CONFIGURATOR': 'aws_configurator',
    'OTEL_TRACES_EXPORTER': 'otlp',
    'OTEL_LOGS_EXPORTER': 'otlp',
    'OTEL_METRICS_EXPORTER': 'none',
    'OTEL_EXPORTER_OTLP_PROTOCOL': 'http/protobuf',
    'OTEL_SERVICE_NAME': OTEL_SERVICE_NAME,
    'OTEL_EXPORTER_OTLP_TRACES_ENDPOINT': f'https://xray.{REGION}.amazonaws.com/v1/traces',
    'OTEL_EXPORTER_OTLP_LOGS_ENDPOINT': f'https://logs.{REGION}.amazonaws.com/v1/logs',
    'OTEL_SEMCONV_STABILITY_OPT_IN': 'gen_ai_tool_definitions',
}

print("Agent Environment Variables:")
print(f"  Application:")
print(f"    AGENT_REGION: {REGION}")
print(f"    GATEWAY_URL: {GATEWAY_URL[:60]}...")
print(f"    MODEL_ID: {agent_env_vars['MODEL_ID']}")
print(f"\n  OpenTelemetry:")
print(f"    OTEL_SERVICE_NAME: {OTEL_SERVICE_NAME}")
print(f"    OTEL_TRACES_EXPORTER: otlp -> X-Ray ({REGION})")
print(f"    OTEL_LOGS_EXPORTER: otlp -> CloudWatch Logs ({REGION})")
print(f"    OTEL_SEMCONV_STABILITY_OPT_IN: gen_ai_tool_definitions")

print("\n\nDeploying Product Catalog Agent to AgentCore Runtime...")
print("This may take several minutes while the container starts up.\n")

runtime_response = create_agent_runtime(
    agentcore_client,
    runtime_name=RUNTIME_NAME,
    role_arn=RUNTIME_ROLE_ARN,
    container_uri=CONTAINER_URI,
    environment_vars=agent_env_vars,
    description='Product Catalog Agent with RBAC and OpenTelemetry observability'
)

if runtime_response:
    RUNTIME_ID = runtime_response['agentRuntimeId']
    RUNTIME_ARN = runtime_response['agentRuntimeArn']
    print(f"\nRuntime deployed successfully!")
    print(f"  Runtime ID: {RUNTIME_ID}")
    print(f"  Runtime ARN: {RUNTIME_ARN}")
    print(f"  Status: {runtime_response.get('status')}")
    print(f"  Observability: Enabled (OTEL traces + GenAI events)")
else:
    print("ERROR: Runtime deployment failed")

Agent Environment Variables:
  Application:
    AGENT_REGION: us-west-2
    GATEWAY_URL: https://ecommerce-workshop-product-gateway-2vhjfcwgfy.gatewa...
    MODEL_ID: global.anthropic.claude-haiku-4-5-20251001-v1:0

  OpenTelemetry:
    OTEL_SERVICE_NAME: ecommerce-workshop-product-catalog-agent
    OTEL_TRACES_EXPORTER: otlp -> X-Ray (us-west-2)
    OTEL_LOGS_EXPORTER: otlp -> CloudWatch Logs (us-west-2)
    OTEL_SEMCONV_STABILITY_OPT_IN: gen_ai_tool_definitions


Deploying Product Catalog Agent to AgentCore Runtime...
This may take several minutes while the container starts up.

Runtime 'ecommerce_workshop_product_catalog_agent' already exists, retrieving...

Runtime deployed successfully!
  Runtime ID: ecommerce_workshop_product_catalog_agent-Nda58K37yj
  Runtime ARN: arn:aws:bedrock-agentcore:us-west-2:534409838809:runtime/ecommerce_workshop_product_catalog_agent-Nda58K37yj
  Status: READY
  Observability: Enabled (OTEL traces + GenAI events)


#### Why Observability Enables Evaluation

The OTEL instrumentation captures detailed traces of every agent interaction — LLM calls, tool invocations, reasoning steps — forming the foundation for Modules 04 and 05.

Key configuration choices:
- **`gen_ai_tool_definitions`**: Captures tool call parameters and results, which `ToolParameterAccuracyEvaluator` and `RBACComplianceEvaluator` depend on
- **`strands.telemetry.tracer` scope**: Module 05's batch evaluation pipeline filters for this scope to find final agent responses
- **`finish_reason: end_turn`**: Marks the final response in a conversation turn — the specific event that triggers evaluation

Without this instrumentation, the agent would work but be a black box. Evaluation requires visibility into the agent's decision-making process, not just its final output.

## Step 10a: Verify CloudWatch Transaction Search

CloudWatch Transaction Search must be enabled to view AgentCore traces and spans in the GenAI Observability dashboard. This is a one-time prerequisite per account/region.

In [18]:
from datetime import datetime, timezone, timedelta
from botocore.exceptions import ClientError

xray_client = boto3.client('xray', region_name=REGION)
logs_client = boto3.client('logs', region_name=REGION)

def check_transaction_search_enabled():
    """Check if CloudWatch Transaction Search is enabled by querying X-Ray."""
    print("Checking CloudWatch Transaction Search status...\n")
    try:
        end_time = datetime.now(timezone.utc)
        start_time = end_time - timedelta(hours=1)
        xray_client.get_trace_summaries(StartTime=start_time, EndTime=end_time, Sampling=True)
        print("CloudWatch Transaction Search is ENABLED")
        return True
    except ClientError as e:
        error_code = e.response['Error']['Code']
        if 'ResourceNotFoundException' in error_code:
            print("CloudWatch Transaction Search is NOT ENABLED")
            print(f"\nTo enable it:")
            print(f"1. Go to: https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#xray:settings/transaction-search")
            print("2. Click 'Edit'")
            print("3. Enable 'Ingest spans as structured logs in OpenTelemetry format'")
            print("4. Click 'Save'")
            return False
        elif 'AccessDenied' in str(e):
            print("Cannot verify (insufficient IAM permissions for X-Ray) - proceeding anyway")
            return True
        else:
            print(f"Unexpected error: {error_code} - proceeding anyway")
            return True
    except Exception as e:
        print(f"Error checking status: {e} - proceeding anyway")
        return True

TRANSACTION_SEARCH_OK = check_transaction_search_enabled()

Checking CloudWatch Transaction Search status...

CloudWatch Transaction Search is ENABLED


## Step 10b: Configure CloudWatch Log Delivery

To enable observability, we must configure CloudWatch Log Delivery to route OTEL telemetry from AgentCore to the appropriate destinations. Without this configuration, the agent emits telemetry but it doesn't appear in CloudWatch.

We configure two delivery types:
1. **APPLICATION_LOGS → CloudWatch Log Group** — Routes GenAI semantic convention events (LLM calls, tool invocations) to a vended log group
2. **TRACES → X-Ray** — Routes OTEL spans to X-Ray for trace visualization

Each delivery requires three components:
- **Delivery Source** — Links the agent runtime as a source of telemetry
- **Delivery Destination** — Specifies where telemetry should go (CWL or X-Ray)
- **Delivery Connection** — Connects the source to the destination

In [31]:
def configure_log_delivery(runtime_id: str, runtime_arn: str, region: str, account_id: str) -> dict:
    """
    Configure CloudWatch Log Delivery for both APPLICATION_LOGS and TRACES.
    
    Includes the prerequisite setup for Transaction Search:
    1. CloudWatch Logs resource policy granting X-Ray write access to aws/spans
    2. UpdateTraceSegmentDestination to enable CloudWatch Logs as trace destination
       (polls until ACTIVE before proceeding)
    3. APPLICATION_LOGS delivery (GenAI events -> CWL vended log group)
    4. TRACES delivery (spans -> X-Ray via aws/spans log group)
    
    Returns dict with delivery configuration results.
    """
    import time as _time
    
    # Extract short suffix for delivery names (must be under 60 chars)
    short_suffix = runtime_id.split('-')[-1] if '-' in runtime_id else runtime_id[-10:]
    
    print(f"Configuring CloudWatch Log Delivery for observability...")
    print(f"  Runtime ID: {runtime_id}")
    print(f"  Short suffix: {short_suffix}")
    
    results = {}
    
    # ============================================================
    # Step 0a: Create CloudWatch Logs resource policy for X-Ray
    # ============================================================
    # X-Ray needs permission to write spans to the aws/spans log group.
    # This is a prerequisite for UpdateTraceSegmentDestination.
    # See: https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability-configure.html
    print(f"\n--- Creating CloudWatch Logs resource policy for X-Ray ---")
    
    resource_policy_document = json.dumps({
        "Version": "2012-10-17",
        "Statement": [
            {
                "Sid": "TransactionSearchXRayAccess",
                "Effect": "Allow",
                "Principal": {
                    "Service": "xray.amazonaws.com"
                },
                "Action": "logs:PutLogEvents",
                "Resource": [
                    f"arn:aws:logs:{region}:{account_id}:log-group:aws/spans:*",
                    f"arn:aws:logs:{region}:{account_id}:log-group:/aws/application-signals/data:*"
                ],
                "Condition": {
                    "ArnLike": {
                        "aws:SourceArn": f"arn:aws:xray:{region}:{account_id}:*"
                    },
                    "StringEquals": {
                        "aws:SourceAccount": account_id
                    }
                }
            }
        ]
    })
    
    try:
        logs_client.put_resource_policy(
            policyName='TransactionSearchXRayAccess',
            policyDocument=resource_policy_document
        )
        print("  Created resource policy: TransactionSearchXRayAccess")
        print("  (Grants xray.amazonaws.com -> logs:PutLogEvents on aws/spans)")
    except ClientError as e:
        if 'already exists' in str(e).lower():
            print("  Resource policy already exists")
        else:
            print(f"  Warning: {e}")
    
    # ============================================================
    # Step 0b: Enable CloudWatch Logs as Trace Segment Destination
    # ============================================================
    # Configure X-Ray to store trace segments in CloudWatch Logs.
    # The API returns Status: PENDING | ACTIVE. We must wait for
    # ACTIVE before CreateDelivery will accept XRAY destinations.
    print(f"\n--- Enabling CloudWatch Logs trace segment destination ---")
    trace_dest_active = False
    try:
        resp = xray_client.update_trace_segment_destination(Destination='CloudWatchLogs')
        status = resp.get('Status', 'UNKNOWN')
        print(f"  UpdateTraceSegmentDestination: Destination=CloudWatchLogs, Status={status}")
        
        if status == 'ACTIVE':
            trace_dest_active = True
        else:
            # Poll GetTraceSegmentDestination until ACTIVE
            max_wait = 120
            poll_interval = 10
            waited = 0
            while waited < max_wait:
                _time.sleep(poll_interval)
                waited += poll_interval
                try:
                    check = xray_client.get_trace_segment_destination()
                    status = check.get('Status', check.get('Destination', {}).get('Status', 'UNKNOWN'))
                    dest = check.get('Destination', 'UNKNOWN')
                    print(f"  Waiting for trace destination to be ACTIVE (status: {status}, waited: {waited}s)...")
                    if status == 'ACTIVE':
                        trace_dest_active = True
                        break
                except Exception as poll_err:
                    print(f"  Poll error: {poll_err}")
            
            if trace_dest_active:
                print("  Trace segment destination is ACTIVE")
            else:
                print(f"  Warning: Trace destination did not reach ACTIVE within {max_wait}s")
                print("  TRACES delivery may fail — try re-running this cell later")
    except ClientError as e:
        error_code = e.response['Error']['Code']
        if 'InvalidRequestException' in error_code:
            # May mean it's already set — check current state
            print("  Trace destination may already be configured, checking...")
            try:
                check = xray_client.get_trace_segment_destination()
                dest = check.get('Destination', 'UNKNOWN')
                status = check.get('Status', 'UNKNOWN')
                print(f"  Current: Destination={dest}, Status={status}")
                trace_dest_active = (status == 'ACTIVE')
            except Exception:
                trace_dest_active = True  # Assume OK if we can't check
        else:
            print(f"  Warning: {e}")
    except Exception as e:
        print(f"  Warning: {e}")

    # ============================================================
    # Step 1: APPLICATION_LOGS delivery (for GenAI events)
    # ============================================================
    print(f"\n--- APPLICATION_LOGS (for GenAI events) ---")
    
    # Create vended log group
    vended_log_group = f"/aws/vendedlogs/bedrock-agentcore/{runtime_id}"
    log_group_arn = f"arn:aws:logs:{region}:{account_id}:log-group:{vended_log_group}"
    
    try:
        logs_client.create_log_group(logGroupName=vended_log_group)
        print(f"  Created vended log group: {vended_log_group}")
    except ClientError as e:
        if e.response['Error']['Code'] == 'ResourceAlreadyExistsException':
            print(f"  Vended log group already exists")
        else:
            print(f"  Error creating log group: {e}")
    results['log_group'] = vended_log_group
    
    # Create logs delivery source
    logs_source_name = f"product-{short_suffix}-logs-src"
    try:
        logs_client.put_delivery_source(
            name=logs_source_name,
            resourceArn=runtime_arn,
            logType='APPLICATION_LOGS'
        )
        print(f"  Created APPLICATION_LOGS delivery source: {logs_source_name}")
    except ClientError as e:
        if 'ConflictException' in str(e) or 'already exists' in str(e).lower():
            print(f"  Logs source already exists")
        else:
            print(f"  Error: {e}")
    results['logs_source'] = logs_source_name
    
    # Create logs delivery destination (CloudWatch Log Group)
    logs_dest_name = f"product-{short_suffix}-logs-dst"
    try:
        logs_client.put_delivery_destination(
            name=logs_dest_name,
            deliveryDestinationType='CWL',
            deliveryDestinationConfiguration={
                'destinationResourceArn': log_group_arn
            }
        )
        print(f"  Created APPLICATION_LOGS delivery destination (CWL)")
    except ClientError as e:
        if 'ConflictException' in str(e) or 'already exists' in str(e).lower():
            print(f"  Logs destination already exists")
        else:
            print(f"  Error: {e}")
    results['logs_destination'] = logs_dest_name
    
    # Create logs delivery connection
    try:
        dest_info = logs_client.get_delivery_destination(name=logs_dest_name)
        dest_arn = dest_info['deliveryDestination']['arn']
        logs_client.create_delivery(
            deliverySourceName=logs_source_name,
            deliveryDestinationArn=dest_arn
        )
        print(f"  Created APPLICATION_LOGS delivery connection")
    except ClientError as e:
        if 'ConflictException' in str(e) or 'already exists' in str(e).lower():
            print(f"  Logs delivery connection already exists")
        else:
            print(f"  Error: {e}")
    
    # ============================================================
    # Step 2: TRACES delivery (for spans in X-Ray)
    # ============================================================
    print(f"\n--- TRACES (for spans in X-Ray) ---")
    
    if not trace_dest_active:
        print("  Skipping TRACES delivery — trace segment destination is not ACTIVE")
        print("  Re-run this cell once the destination is active")
        return results
    
    # Create traces delivery source
    traces_source_name = f"product-{short_suffix}-trace-src"
    try:
        logs_client.put_delivery_source(
            name=traces_source_name,
            resourceArn=runtime_arn,
            logType='TRACES'
        )
        print(f"  Created TRACES delivery source: {traces_source_name}")
    except ClientError as e:
        if 'ConflictException' in str(e) or 'already exists' in str(e).lower():
            print(f"  Traces source already exists")
        else:
            print(f"  Error: {e}")
    results['traces_source'] = traces_source_name
    
    # Create traces delivery destination (X-Ray)
    traces_dest_name = f"product-{short_suffix}-trace-dst"
    try:
        logs_client.put_delivery_destination(
            name=traces_dest_name,
            deliveryDestinationType='XRAY'
        )
        print(f"  Created TRACES delivery destination (XRAY)")
    except ClientError as e:
        if 'ConflictException' in str(e) or 'already exists' in str(e).lower():
            print(f"  Traces destination already exists")
        else:
            print(f"  Error: {e}")
    results['traces_destination'] = traces_dest_name
    
    # Create traces delivery connection
    try:
        dest_info = logs_client.get_delivery_destination(name=traces_dest_name)
        dest_arn = dest_info['deliveryDestination']['arn']
        logs_client.create_delivery(
            deliverySourceName=traces_source_name,
            deliveryDestinationArn=dest_arn
        )
        print(f"  Created TRACES delivery connection")
    except ClientError as e:
        if 'ConflictException' in str(e) or 'already exists' in str(e).lower():
            print(f"  Traces delivery connection already exists")
        else:
            print(f"  Error creating TRACES delivery: {e}")
    
    return results


# Configure log delivery for the product catalog agent runtime
LOG_DELIVERY_CONFIG = configure_log_delivery(RUNTIME_ID, RUNTIME_ARN, REGION, ACCOUNT_ID)

print(f"\n{'='*60}")
print("LOG DELIVERY SUMMARY")
print(f"{'='*60}")
print(f"  APPLICATION_LOGS -> {LOG_DELIVERY_CONFIG.get('log_group', 'N/A')}")
if 'traces_source' in LOG_DELIVERY_CONFIG:
    print(f"  TRACES -> X-Ray (aws/spans log group)")
else:
    print(f"  TRACES -> NOT CONFIGURED (re-run this cell)")
print(f"\nObservability pipeline is configured. Traces will appear in CloudWatch")
print(f"GenAI Observability dashboard after the first agent invocation.")

Configuring CloudWatch Log Delivery for observability...
  Runtime ID: ecommerce_workshop_product_catalog_agent-Nda58K37yj
  Short suffix: Nda58K37yj

--- Creating CloudWatch Logs resource policy for X-Ray ---
  Created resource policy: TransactionSearchXRayAccess
  (Grants xray.amazonaws.com -> logs:PutLogEvents on aws/spans)

--- Enabling CloudWatch Logs trace segment destination ---
  Trace destination may already be configured, checking...
  Current: Destination=CloudWatchLogs, Status=ACTIVE

--- APPLICATION_LOGS (for GenAI events) ---
  Vended log group already exists
  Created APPLICATION_LOGS delivery source: product-Nda58K37yj-logs-src
  Created APPLICATION_LOGS delivery destination (CWL)
  Created APPLICATION_LOGS delivery connection

--- TRACES (for spans in X-Ray) ---
  Created TRACES delivery source: product-Nda58K37yj-trace-src
  Created TRACES delivery destination (XRAY)
  Created TRACES delivery connection

LOG DELIVERY SUMMARY
  APPLICATION_LOGS -> /aws/vendedlogs/bedro

## Step 11: Test Production Agent - Customer Role

Now let's test the deployed agent with a **customer** JWT token. The customer should:
- Be able to use 6 read-only tools (search, details, inventory, recommendations, compare, return policy)
- NOT be able to use 5 admin tools (create, update, delete, update inventory, update pricing)

RBAC is enforced at two levels:
1. The agent filters tools based on JWT claims (application-level)
2. The Gateway interceptor blocks unauthorized tool calls (infrastructure-level)

In [32]:
from utils import invoke_agent_runtime

agentcore_runtime_client = boto3.client('bedrock-agentcore', region_name=REGION)

# Test with customer token
customer_session_id = f"customer-{str(uuid.uuid4())}"

print("=" * 60)
print("Customer Test 1: Product Search (ALLOWED)")
print("=" * 60)

result = invoke_agent_runtime(
    agentcore_runtime_client,
    RUNTIME_ARN,
    customer_session_id,
    {
        'prompt': 'Search for wireless headphones under $100',
        'bearer_token': customer_tokens.get('id_token', ''),
        'access_token': customer_tokens.get('access_token', ''),
        'session_id': customer_session_id
    }
)

print(f"\nStatus: {result.get('status')}")
print(f"Role: {result.get('metadata', {}).get('role')}")
print(f"Tools available: {result.get('metadata', {}).get('tools_available')}")
print(f"Tools used: {result.get('metadata', {}).get('tools_used')}")
print(f"\nResponse:\n{result.get('response', result.get('error', 'No response'))}")

Customer Test 1: Product Search (ALLOWED)

Status: success
Role: customer
Tools available: 6
Tools used: ['search_products', 'get_product_recommendations']

Response:
Great! I found wireless headphones for you. Here are the results under $100:

**1. Wireless Bluetooth Headphones - $79.99** ✓ Under Budget
- **Product ID:** PROD-001
- **Price:** $79.99
- **Rating:** 4.5/5 stars
- **In Stock:** Yes
- **Description:** Premium over-ear wireless headphones with 40-hour battery life, active noise cancellation, and comfortable memory foam ear cushions.

---

**2. Noise Canceling Earbuds - $149.99** (Over budget)
- **Product ID:** PROD-055
- **Price:** $149.99
- **Rating:** 4.5/5 stars
- **In Stock:** Yes
- **Description:** True wireless earbuds with adaptive noise cancellation, transparency mode, and 8-hour battery life per charge.

The **Wireless Bluetooth Headphones (PROD-001)** is the perfect match for your budget at $79.99! It offers excellent features including active noise cancellation a

In [33]:
# Customer Test 2: Attempt admin action (SHOULD BE REFUSED)
print("=" * 60)
print("Customer Test 2: Create Product (SHOULD BE REFUSED)")
print("=" * 60)

result = invoke_agent_runtime(
    agentcore_runtime_client,
    RUNTIME_ARN,
    customer_session_id,
    {
        'prompt': 'Create a new product called Super Speaker for $299.99 in Audio category',
        'bearer_token': customer_tokens.get('id_token', ''),
        'access_token': customer_tokens.get('access_token', ''),
        'session_id': customer_session_id
    }
)

print(f"\nStatus: {result.get('status')}")
print(f"Role: {result.get('metadata', {}).get('role')}")
print(f"Tools available: {result.get('metadata', {}).get('tools_available')}")
print(f"\nResponse:\n{result.get('response', result.get('error', 'No response'))}")
print("\n(Customer should be refused - create_product is admin only)")

Customer Test 2: Create Product (SHOULD BE REFUSED)

Status: success
Role: customer
Tools available: 6

Response:
I appreciate you reaching out, but I'm unable to create new products in the catalog. That operation requires administrator privileges, which I don't have access to.

My capabilities are limited to:
- **Searching** for existing products
- **Viewing** detailed product information
- **Checking** inventory and stock availability
- **Recommending** products based on your preferences
- **Comparing** products side by side
- **Explaining** return policies

To add a new product like "Super Speaker" to the catalog, you would need to:
1. Contact your store administrator or manager
2. Access the admin panel or product management system directly
3. Use the appropriate administrative tools for product creation

Is there anything else I can help you with regarding existing products in our catalog? For example, I could help you search for similar audio products or compare existing speakers

# Admin Test 1: Search (admin has all customer tools too)

In [34]:
admin_session_id = f"admin-{str(uuid.uuid4())}"

print("=" * 60)
print("Admin Test 1: Product Search (ALLOWED)")
print("=" * 60)

result = invoke_agent_runtime(
    agentcore_runtime_client,
    RUNTIME_ARN,
    admin_session_id,
    {
        'prompt': 'Search for headphone products',
        'bearer_token': admin_tokens.get('id_token', ''),
        'access_token': admin_tokens.get('access_token', ''),
        'session_id': admin_session_id
    }
)

print(f"\nStatus: {result.get('status')}")
print(f"Role: {result.get('metadata', {}).get('role')}")
print(f"Tools available: {result.get('metadata', {}).get('tools_available')}")
print(f"Tools used: {result.get('metadata', {}).get('tools_used')}")
print(f"\nResponse:\n{result.get('response', result.get('error', 'No response'))}")

Admin Test 1: Product Search (ALLOWED)

Status: success
Role: admin
Tools available: 11
Tools used: ['search_products']

Response:
I found **1 headphone product** in the catalog:

**Wireless Bluetooth Headphones** (PROD-001)
- **Price:** $79.99
- **Category:** Audio
- **In Stock:** Yes
- **Rating:** 4.5/5
- **Description:** Premium over-ear wireless headphones with 40-hour battery life, active noise cancellation, and comfortable memory foam ear cushions.

Would you like me to get more detailed information about this product, or would you like to search for something more specific?


In [35]:
# Admin Test 2: Set a sale price (ALLOWED for admin)
print("=" * 60)
print("Admin Test 2: Update Pricing (ALLOWED)")
print("=" * 60)

result = invoke_agent_runtime(
    agentcore_runtime_client,
    RUNTIME_ARN,
    admin_session_id,
    {
        'prompt': 'Set a sale price of $99.99 for PROD-201 (regular price stays $129.99) until 2025-06-30',
        'bearer_token': admin_tokens.get('id_token', ''),
        'access_token': admin_tokens.get('access_token', ''),
        'session_id': admin_session_id
    }
)

print(f"\nStatus: {result.get('status')}")
print(f"Tools used: {result.get('metadata', {}).get('tools_used')}")
print(f"\nResponse:\n{result.get('response', result.get('error', 'No response'))}")

Admin Test 2: Update Pricing (ALLOWED)

Status: success
Tools used: ['update_pricing']

Response:
It looks like product PROD-201 doesn't exist in the catalog. Could you please verify the product ID? 

Would you like me to:
1. Search for a similar product ID?
2. Check what products are currently available?
3. Try a different product ID?

Please provide the correct product ID and I'll set up the sale price for you.


In [36]:
# Admin Test 3: Clean up - discontinue the test product
print("=" * 60)
print("Admin Test 3: Delete Product (ALLOWED - soft delete)")
print("=" * 60)

result = invoke_agent_runtime(
    agentcore_runtime_client,
    RUNTIME_ARN,
    admin_session_id,
    {
        'prompt': 'Discontinue product PROD-201',
        'bearer_token': admin_tokens.get('id_token', ''),
        'access_token': admin_tokens.get('access_token', ''),
        'session_id': admin_session_id
    }
)

print(f"\nStatus: {result.get('status')}")
print(f"Tools used: {result.get('metadata', {}).get('tools_used')}")
print(f"\nResponse:\n{result.get('response', result.get('error', 'No response'))}")

Admin Test 3: Delete Product (ALLOWED - soft delete)

Status: success
Tools used: ['delete_product']

Response:
The product PROD-201 could not be found in the catalog. This product either:
- Doesn't exist in the system
- Has already been discontinued
- May have a different product ID

Would you like me to:
1. Search for a similar product ID?
2. List available products in a specific category?
3. Verify the correct product ID?

Please provide the correct product ID if you have it, and I'll discontinue it for you.


## Step 13: Validate Observability Traces

After running test invocations, let's verify that traces and GenAI events are being captured in CloudWatch. Traces typically take 1-2 minutes to appear after invocation.

### What You'll See in CloudWatch GenAI Observability

- **invoke_agent** spans — Agent invocation with input/output
- **chat** spans — LLM model calls with token usage
- **execute_tool** spans — Tool calls with inputs/outputs
- **execute_event_loop_cycle** spans — Agent reasoning steps
- **gen_ai events** — Detailed GenAI semantic convention events

In [37]:
# Wait for traces to propagate to CloudWatch
print("Waiting 30 seconds for traces to propagate to CloudWatch...")
print("(Traces typically take 1-2 minutes to appear after invocation)")
time.sleep(30)

def query_spans_log_group(time_range_minutes=30):
    """Query the aws/spans log group for OTEL spans."""
    print("\nQuerying aws/spans log group for OTEL traces...")
    results = {'total_spans': 0, 'span_types': {}, 'services': set()}
    
    end_time = int(datetime.now(timezone.utc).timestamp() * 1000)
    start_time = int((datetime.now(timezone.utc) - timedelta(minutes=time_range_minutes)).timestamp() * 1000)
    
    for log_group in ["aws/spans", "aws/spans/default"]:
        try:
            streams_response = logs_client.describe_log_streams(
                logGroupName=log_group, orderBy='LastEventTime', descending=True, limit=10
            )
            streams = streams_response.get('logStreams', [])
            if not streams:
                continue
            
            print(f"  Found {len(streams)} log streams in {log_group}")
            
            for stream in streams:
                events_response = logs_client.get_log_events(
                    logGroupName=log_group,
                    logStreamName=stream['logStreamName'],
                    startTime=start_time, endTime=end_time, limit=200
                )
                for event in events_response.get('events', []):
                    try:
                        span_data = json.loads(event.get('message', ''))
                        results['total_spans'] += 1
                        span_name = span_data.get('name', 'unknown')
                        results['span_types'][span_name] = results['span_types'].get(span_name, 0) + 1
                        scope = span_data.get('scope', {})
                        if scope.get('name'):
                            results['services'].add(scope['name'])
                    except json.JSONDecodeError:
                        pass
            
            if results['total_spans'] > 0:
                print(f"  Found {results['total_spans']} spans in {log_group}")
                break
        except logs_client.exceptions.ResourceNotFoundException:
            pass
        except Exception as e:
            print(f"  Error querying {log_group}: {str(e)[:80]}")
    
    return results


def query_vendedlogs_for_genai_events(runtime_id, time_range_minutes=30):
    """Query vendedlogs for GenAI events (APPLICATION_LOGS delivery)."""
    print("\nQuerying vendedlogs for GenAI events...")
    log_group = f"/aws/vendedlogs/bedrock-agentcore/{runtime_id}"
    
    end_time = int(datetime.now(timezone.utc).timestamp() * 1000)
    start_time = int((datetime.now(timezone.utc) - timedelta(minutes=time_range_minutes)).timestamp() * 1000)
    
    try:
        streams_response = logs_client.describe_log_streams(
            logGroupName=log_group, orderBy='LastEventTime', descending=True, limit=5
        )
        total_events = 0
        for stream in streams_response.get('logStreams', []):
            events_response = logs_client.get_log_events(
                logGroupName=log_group,
                logStreamName=stream['logStreamName'],
                startTime=start_time, endTime=end_time, limit=100
            )
            total_events += len(events_response.get('events', []))
        
        if total_events > 0:
            print(f"  Found {total_events} GenAI events in {log_group}")
        return total_events
    except logs_client.exceptions.ResourceNotFoundException:
        print(f"  Log group not found yet: {log_group}")
        return 0
    except Exception as e:
        print(f"  Error: {str(e)[:80]}")
        return 0


# Run validation
print("=" * 60)
print("VALIDATING OBSERVABILITY DATA IN CLOUDWATCH")
print("=" * 60)

spans_results = query_spans_log_group(time_range_minutes=30)
genai_events = query_vendedlogs_for_genai_events(RUNTIME_ID, time_range_minutes=30)

# Summary
print(f"\n{'='*60}")
print("OBSERVABILITY VALIDATION SUMMARY")
print(f"{'='*60}")

spans_count = spans_results.get('total_spans', 0)
print(f"\n  OTEL Spans (aws/spans): {spans_count} spans")
if spans_results.get('span_types'):
    print("  Span types found:")
    for span_type, count in list(spans_results['span_types'].items())[:8]:
        print(f"    - {span_type}: {count}")

print(f"\n  GenAI Events (vendedlogs): {genai_events} events")

total_data = spans_count + genai_events
print(f"\n  Total Observability Data: {total_data} records")

print(f"\n  View in CloudWatch GenAI Observability Dashboard:")
print(f"  https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#genai-observability:bedrockAgentCore")

if total_data > 0:
    print("\nObservability is working! Traces and GenAI events are being captured.")
else:
    print("\nNo observability data found yet. Possible reasons:")
    print("  - Data may still be propagating (wait 2-3 more minutes)")
    print("  - Log delivery configuration may need time to activate")
    print("  - Check the CloudWatch console directly")

Waiting 30 seconds for traces to propagate to CloudWatch...
(Traces typically take 1-2 minutes to appear after invocation)
VALIDATING OBSERVABILITY DATA IN CLOUDWATCH

Querying aws/spans log group for OTEL traces...
  Found 1 log streams in aws/spans
  Found 62 spans in aws/spans

Querying vendedlogs for GenAI events...
  Found 18 GenAI events in /aws/vendedlogs/bedrock-agentcore/ecommerce_workshop_product_catalog_agent-Nda58K37yj

OBSERVABILITY VALIDATION SUMMARY

  OTEL Spans (aws/spans): 62 spans
  Span types found:
    - POST: 20
    - chat global.anthropic.claude-haiku-4-5-20251001-v1:0: 9
    - chat: 9
    - execute_tool ProductTools___get_product_recommendations: 1
    - execute_tool ProductTools___search_products: 2
    - execute_event_loop_cycle: 9
    - invoke_agent Strands Agents: 5
    - POST /invocations: 5

  GenAI Events (vendedlogs): 18 events

  Total Observability Data: 80 records

  View in CloudWatch GenAI Observability Dashboard:
  https://us-west-2.console.aws.ama

In [38]:
from utils import save_config

# Build agent configuration
agent_config = {
    'runtime_arn': RUNTIME_ARN,
    'runtime_id': RUNTIME_ID,
    'gateway_id': GATEWAY_ID,
    'gateway_url': GATEWAY_URL,
    'region': REGION,
    'user_pool_id': USER_POOL_ID,
    'user_client_id': USER_CLIENT_ID,
    'customer_email': CUSTOMER_EMAIL,
    'admin_email': ADMIN_EMAIL,
    'test_password': TEST_PASSWORD,
    'model_id': 'global.anthropic.claude-haiku-4-5-20251001-v1:0',
    'otel_service_name': OTEL_SERVICE_NAME
}

save_config(agent_config, 'streamlit_app/agent_config.json')

print("\nConfiguration saved for Streamlit app.")
print("\nTo launch the chat interface:")
print("  cd streamlit_app")
print("  pip install streamlit")
print("  streamlit run app.py")
print("\nThe app will be available at http://localhost:8501")

Saved config to: streamlit_app/agent_config.json

Configuration saved for Streamlit app.

To launch the chat interface:
  cd streamlit_app
  pip install streamlit
  streamlit run app.py

The app will be available at http://localhost:8501


In [39]:
print("=" * 60)
print("DEPLOYMENT COMPLETE!")
print("=" * 60)

print(f"\nInfrastructure:")
print(f"  AgentCore Runtime: {RUNTIME_ARN}")
print(f"  AgentCore Gateway: {GATEWAY_URL}")
print(f"  RBAC Interceptor:  {INTERCEPTOR_ARN}")
print(f"  Cognito Pool:      {USER_POOL_ID}")

print(f"\nMCP Tools (11 total):")
print(f"  Read tools (6):  search_products, get_product_details, check_inventory,")
print(f"                   get_product_recommendations, compare_products, get_return_policy")
print(f"  Admin tools (5): create_product, update_product, delete_product,")
print(f"                   update_inventory, update_pricing")

print(f"\nRBAC:")
print(f"  Customer ({CUSTOMER_EMAIL}): 6 read-only tools")
print(f"  Admin ({ADMIN_EMAIL}):    11 tools (6 read + 5 admin)")
print(f"  Enforcement: Agent-side filtering + Gateway interceptor")

print(f"\nObservability:")
print(f"  OTEL Service Name: {OTEL_SERVICE_NAME}")
print(f"  Traces: X-Ray via OTLP")
print(f"  GenAI Events: CloudWatch vendedlogs")
print(f"  Dashboard: https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#genai-observability:bedrockAgentCore")

print(f"\nStreamlit App: streamlit_app/app.py")
print("=" * 60)

DEPLOYMENT COMPLETE!

Infrastructure:
  AgentCore Runtime: arn:aws:bedrock-agentcore:us-west-2:534409838809:runtime/ecommerce_workshop_product_catalog_agent-Nda58K37yj
  AgentCore Gateway: https://ecommerce-workshop-product-gateway-2vhjfcwgfy.gateway.bedrock-agentcore.us-west-2.amazonaws.com/mcp
  RBAC Interceptor:  arn:aws:lambda:us-west-2:534409838809:function:ecommerce-workshop-rbac-interceptor
  Cognito Pool:      us-west-2_zZFgcKaqx

MCP Tools (11 total):
  Read tools (6):  search_products, get_product_details, check_inventory,
                   get_product_recommendations, compare_products, get_return_policy
  Admin tools (5): create_product, update_product, delete_product,
                   update_inventory, update_pricing

RBAC:
  Customer (john.customer@example.com): 6 read-only tools
  Admin (alice.admin@example.com):    11 tools (6 read + 5 admin)
  Enforcement: Agent-side filtering + Gateway interceptor

Observability:
  OTEL Service Name: ecommerce-workshop-product-catal

In [40]:
# Save deployment info for subsequent modules (Module 04: Evaluations)
deployment_info = {
    'runtime_arn': RUNTIME_ARN,
    'runtime_id': RUNTIME_ID,
    'runtime_name': RUNTIME_NAME,
    'gateway_id': GATEWAY_ID,
    'gateway_url': GATEWAY_URL,
    'user_pool_id': USER_POOL_ID,
    'user_client_id': USER_CLIENT_ID,
    'otel_service_name': OTEL_SERVICE_NAME,
    'region': REGION
}

%store deployment_info
%store REGION
print("Deployment information saved for subsequent modules (Module 04: Evaluations)")

Stored 'deployment_info' (dict)
Stored 'REGION' (str)
Deployment information saved for subsequent modules (Module 04: Evaluations)


---

## Module 3 Summary

You deployed the Product Catalog Agent from Module 01 to production using Amazon Bedrock AgentCore, **with full observability enabled from day one**.

### What Was Created

| Component | Description |
|-----------|-------------|
| **Product Tools Lambda** | 11 MCP tools (6 read + 5 admin) as a single Lambda |
| **RBAC Interceptor Lambda** | Gateway interceptor for infrastructure-level access control |
| **Cognito User Pool** | User management with `customer` and `admin` groups |
| **AgentCore Gateway** | MCP endpoint with JWT auth and RBAC interceptor |
| **ECR Repository** | Container image for the Product Catalog Agent |
| **AgentCore Runtime** | Managed hosting with OpenTelemetry observability |
| **CloudWatch Log Delivery** | APPLICATION_LOGS (GenAI events) + TRACES (X-Ray spans) |

### Defense-in-Depth RBAC

| Layer | Mechanism | What It Does |
|-------|-----------|-------------|
| **Application** | Agent tool filtering | Removes admin tools from customer agent (same as Module 01) |
| **Infrastructure** | Gateway interceptor | Blocks admin tool calls and hides admin tools in discovery |

### Observability

| Component | Description |
|-----------|-------------|
| **OpenTelemetry Instrumentation** | `opentelemetry-instrument` wraps agent for auto-instrumentation |
| **AWS OTEL Distro** | `aws-opentelemetry-distro` for CloudWatch/X-Ray integration |
| **GenAI Events** | Detailed LLM calls, tool invocations via `gen_ai_tool_definitions` |
| **CloudWatch Log Delivery** | Routes APPLICATION_LOGS to CWL and TRACES to X-Ray |
| **GenAI Observability Dashboard** | Aggregated view of traces, spans, and GenAI events |

### Key Files

| File | Description |
|------|-------------|
| `lambda_tools/product_tools_lambda.py` | 11 product tools for Gateway Lambda target |
| `lambda_tools/rbac_interceptor_lambda.py` | RBAC interceptor (REQUEST + RESPONSE) |
| `agents/product_catalog_agent.py` | Production agent with JWT role extraction |
| `agents/Dockerfile` | ARM64 container with OTEL instrumentation |
| `agents/requirements.txt` | Dependencies including `strands-agents[otel]` and `aws-opentelemetry-distro` |
| `utils.py` | Deployment utility functions |
| `streamlit_app/app.py` | Chat UI with Cognito login |

### Prototype to Production Mapping

| Module 01 Pattern | Module 03 Implementation |
|-------------------|-------------------------|
| `UserSession(role='customer')` | JWT with `cognito:groups: ['customer']` |
| `UserSession(role='admin')` | JWT with `cognito:groups: ['admin']` |
| `MCPClient(stdio_client(...))` | `MCPClient(streamablehttp_client(...))` with SigV4 |
| `mcp_server.py` (local process) | `product_tools_lambda.py` (Lambda behind Gateway) |
| Agent-side tool filtering only | Agent filtering + Gateway interceptor |
| No observability | Full OTEL tracing + GenAI events in CloudWatch |

### Next Steps

1. Launch the Streamlit app to test the chat interface with different user roles
2. View agent traces in the **CloudWatch GenAI Observability Dashboard**
3. Proceed to **Module 04: Evaluations** to set up online and on-demand evaluation

---

## Cleanup (Optional)

Run the cells below to delete all resources created in this module. **Only run this when you're done with the workshop.**

In [29]:
# UNCOMMENT AND RUN TO CLEAN UP ALL RESOURCES

# from utils import delete_agent_runtime, delete_gateway
#
# print("Cleaning up Module 03 resources...\n")
#
# # 1. Delete AgentCore Runtime
# print("Deleting AgentCore Runtime...")
# delete_agent_runtime(agentcore_client, RUNTIME_ID)
#
# # 2. Delete AgentCore Gateway (and targets)
# print("\nDeleting AgentCore Gateway...")
# delete_gateway(gateway_client, GATEWAY_ID)
#
# # 3. Delete Lambda functions
# print("\nDeleting Lambda functions...")
# lambda_client = boto3.client('lambda', region_name=REGION)
# for func_name in [f'{WORKSHOP_PREFIX}-product-tools', f'{WORKSHOP_PREFIX}-rbac-interceptor']:
#     try:
#         lambda_client.delete_function(FunctionName=func_name)
#         print(f"  Deleted: {func_name}")
#     except Exception as e:
#         print(f"  Error deleting {func_name}: {e}")
#
# # 4. Delete Cognito User Pool
# print("\nDeleting Cognito User Pool...")
# try:
#     cognito_client.delete_user_pool_domain(Domain=DOMAIN_PREFIX, UserPoolId=USER_POOL_ID)
#     cognito_client.delete_user_pool(UserPoolId=USER_POOL_ID)
#     print(f"  Deleted pool: {USER_POOL_ID}")
# except Exception as e:
#     print(f"  Error: {e}")
#
# # 5. Delete ECR repository
# print("\nDeleting ECR repository...")
# try:
#     ecr_client.delete_repository(repositoryName=ECR_REPO_NAME, force=True)
#     print(f"  Deleted: {ECR_REPO_NAME}")
# except Exception as e:
#     print(f"  Error: {e}")
#
# # 6. Delete IAM roles
# print("\nDeleting IAM roles...")
# for role_name in [LAMBDA_ROLE_NAME, GATEWAY_ROLE_NAME, RUNTIME_ROLE_NAME]:
#     try:
#         # Detach managed policies
#         policies = iam_client.list_attached_role_policies(RoleName=role_name)
#         for policy in policies['AttachedPolicies']:
#             iam_client.detach_role_policy(RoleName=role_name, PolicyArn=policy['PolicyArn'])
#         # Delete inline policies
#         inline = iam_client.list_role_policies(RoleName=role_name)
#         for policy_name in inline['PolicyNames']:
#             iam_client.delete_role_policy(RoleName=role_name, PolicyName=policy_name)
#         iam_client.delete_role(RoleName=role_name)
#         print(f"  Deleted: {role_name}")
#     except Exception as e:
#         print(f"  Error deleting {role_name}: {e}")
#
# print("\nCleanup complete!")

---

## Module 3 Summary

You deployed the Product Catalog Agent from Module 01 to production using Amazon Bedrock AgentCore.

### What Was Created

| Component | Description |
|-----------|-------------|
| **Product Tools Lambda** | 11 MCP tools (6 read + 5 admin) as a single Lambda |
| **RBAC Interceptor Lambda** | Gateway interceptor for infrastructure-level access control |
| **Cognito User Pool** | User management with `customer` and `admin` groups |
| **AgentCore Gateway** | MCP endpoint with JWT auth and RBAC interceptor |
| **ECR Repository** | Container image for the Product Catalog Agent |
| **AgentCore Runtime** | Managed hosting for the containerized agent |

### Defense-in-Depth RBAC

| Layer | Mechanism | What It Does |
|-------|-----------|-------------|
| **Application** | Agent tool filtering | Removes admin tools from customer agent (same as Module 01) |
| **Infrastructure** | Gateway interceptor | Blocks admin tool calls and hides admin tools in discovery |

### Key Files

| File | Description |
|------|-------------|
| `lambda_tools/product_tools_lambda.py` | 11 product tools for Gateway Lambda target |
| `lambda_tools/rbac_interceptor_lambda.py` | RBAC interceptor (REQUEST + RESPONSE) |
| `agents/product_catalog_agent.py` | Production agent with JWT role extraction |
| `agents/Dockerfile` | ARM64 container for AgentCore Runtime |
| `utils.py` | Deployment utility functions |
| `streamlit_app/app.py` | Chat UI with Cognito login |

### Prototype to Production Mapping

| Module 01 Pattern | Module 03 Implementation |
|-------------------|-------------------------|
| `UserSession(role='customer')` | JWT with `cognito:groups: ['customer']` |
| `UserSession(role='admin')` | JWT with `cognito:groups: ['admin']` |
| `MCPClient(stdio_client(...))` | `MCPClient(streamablehttp_client(...))` with SigV4 |
| `mcp_server.py` (local process) | `product_tools_lambda.py` (Lambda behind Gateway) |
| Agent-side tool filtering only | Agent filtering + Gateway interceptor |

### Next Steps

1. Launch the Streamlit app to test the chat interface with different user roles
2. Monitor agent performance in CloudWatch logs
3. Explore observability with OpenTelemetry traces